In [7]:
from pathlib import Path
from collections import defaultdict, Counter
from PIL import Image
from PIL.JpegImagePlugin import get_sampling
import statistics
import os
import shutil

In [5]:
def jpeg_quality(img):
    """Estimate JPEG quality from quantization tables (0-100, None if not JPEG)."""
    qt = getattr(img, "quantization", None)
    if not qt:
        return None
    # Standard IJG-based estimate
    total = sum(sum(t) for t in qt.values())
    count = sum(len(t) for t in qt.values())
    avg = total / count
    # Rough inverse mapping of the standard JPEG quality scaling
    if avg <= 0:
        return 100
    q = 100 - avg * 100 / 255  # crude but monotonic
    return round(max(0, min(100, q)), 1)

In [6]:
def detect_bias(parent, subfolders):
    exts = {".jpg", ".jpeg", ".png", ".webp", ".bmp", ".gif", ".tiff", ".tif"}
    for sf in subfolders:
        folder = parent / sf
        if not folder.exists():
            print(f"[MISSING] {sf}")
            continue

        formats = defaultdict(int)
        sizes = defaultdict(int)
        modes = defaultdict(int)
        jpeg_qualities = []
        n = 0

        for f in folder.rglob("*"):
            if f.suffix.lower() not in exts:
                continue
            try:
                with Image.open(f) as img:
                    n += 1
                    formats[img.format] += 1
                    sizes[img.size] += 1
                    modes[img.mode] += 1
                    if img.format == "JPEG":
                        q = jpeg_quality(img)
                        if q is not None:
                            jpeg_qualities.append(q)
            except Exception as e:
                print(f"  [ERROR] {f}: {e}")

        print(f"\n=== {sf} ({n} images) ===")
        print("Formats:", dict(formats))
        print("Modes:", dict(modes))
        print("Sizes (WxH: count):")
        for size, count in sorted(sizes.items(), key=lambda x: -x[1]):
            print(f"    {size[0]}x{size[1]}: {count}")
        if jpeg_qualities:
            import statistics
            print(f"JPEG quality est.: min={min(jpeg_qualities)}, "
                f"max={max(jpeg_qualities)}, "
                f"mean={round(statistics.mean(jpeg_qualities),1)}")

In [7]:
parent = Path("/ceph/tischuet/replication_data/RAISE_1k")  # set to your parent folder
subfolders = ["TIFF"]

detect_bias(parent, subfolders)


=== TIFF (1000 images) ===
Formats: {'TIFF': 1000}
Modes: {'RGB': 1000}
Sizes (WxH: count):
    4928x3264: 571
    4288x2848: 177
    3264x4928: 158
    2848x4288: 86
    3008x2000: 6
    2000x3008: 2


In [8]:
parent = Path("/ceph/tischuet/replication_data/New-Generator")  # set to your parent folder
subfolders = ["dalle3", "firefly", "flux", "midjourney-v5", "real", "sd3", "sdxl"]

detect_bias(parent, subfolders)


=== dalle3 (1000 images) ===
Formats: {'PNG': 1000}
Modes: {'RGBA': 1000}
Sizes (WxH: count):
    1024x1024: 841
    1792x1024: 85
    1024x1792: 74

=== firefly (1000 images) ===
Formats: {'JPEG': 1000}
Modes: {'RGB': 1000}
Sizes (WxH: count):
    2304x1792: 271
    1792x2304: 252
    2048x2048: 243
    2688x1536: 234
JPEG quality est.: min=97.2, max=97.2, mean=97.2

=== flux (5000 images) ===
Formats: {'PNG': 5000}
Modes: {'RGB': 5000}
Sizes (WxH: count):
    512x512: 5000

=== midjourney-v5 (1000 images) ===
Formats: {'PNG': 1000}
Modes: {'RGB': 1000}
Sizes (WxH: count):
    1360x896: 571
    1344x896: 182
    896x1360: 158
    896x1344: 88
    1024x1024: 1

=== real (5000 images) ===
Formats: {'JPEG': 5000}
Modes: {'RGB': 4990, 'L': 10}
Sizes (WxH: count):
    640x480: 1061
    640x427: 606
    480x640: 336
    500x375: 240
    640x426: 215
    427x640: 171
    640x428: 161
    640x425: 146
    612x612: 106
    375x500: 94
    500x333: 80
    426x640: 73
    640x640: 54
    640x36

Note: Firefly images are already compressed in their original source. Synthbuster dataset download

In [13]:
parent = Path("/home/tischuet/zero-shot-deepfake-detection/datasets/synthbuster")  # set to your parent folder
subfolders = ["firefly"]

detect_bias(parent, subfolders)


=== firefly (1000 images) ===
Formats: {'JPEG': 1000}
Modes: {'RGB': 1000}
Sizes (WxH: count):
    2304x1792: 271
    1792x2304: 252
    2048x2048: 243
    2688x1536: 234
JPEG quality est.: min=97.2, max=97.2, mean=97.2


In [10]:
folder = Path("/ceph/tischuet/replication_data/New-Generator/real")
exts = {".jpg", ".jpeg", ".png", ".webp", ".bmp", ".gif", ".tiff", ".tif"}
target_q = 97.2  # exact quality value to count

qualities = Counter()
files_by_quality = {}  # keep a few example filenames per quality
n = 0
non_jpeg = 0

for f in folder.rglob("*"):
    if f.suffix.lower() not in exts:
        continue
    try:
        with Image.open(f) as img:
            n += 1
            if img.format == "JPEG":
                q = jpeg_quality(img)
                qualities[q] += 1
                files_by_quality.setdefault(q, []).append(f.name)
            else:
                non_jpeg += 1
    except Exception as e:
        print(f"[ERROR] {f}: {e}")

print(f"Total images: {n}, non-JPEG: {non_jpeg}")
print(f"Count with quality == {target_q}: {qualities[target_q]}")
print("\nQuality distribution (quality: count):")
for q, c in sorted(qualities.items()):
    print(f"  {q}: {c}   (e.g. {files_by_quality[q][:3]})")


Total images: 5000, non-JPEG: 0
Count with quality == 97.2: 0

Quality distribution (quality: count):
  88.7: 46   (e.g. ['000000219271.jpg', '000000149222.jpg', '000000093261.jpg'])
  94.3: 1414   (e.g. ['000000256868.jpg', '000000013546.jpg', '000000404568.jpg'])
  97.7: 3530   (e.g. ['000000217060.jpg', '000000317024.jpg', '000000308753.jpg'])
  98.2: 10   (e.g. ['000000209222.jpg', '000000141671.jpg', '000000061418.jpg'])


In [ ]:
parent = Path("/ceph/tischuet/replication_data/ForenSynths")  # set to your parent folder
subfolders = ["biggan", "cyclegan", "gaugan", "progan", "stargan", "stylegan", "stylegan2"]
exts = {".jpg", ".jpeg", ".png", ".webp", ".bmp", ".gif", ".tiff", ".tif"}


def which_split(path):
    parts = {p.lower() for p in path.parts}
    if "0_real" in parts:
        return "real"
    if "1_fake" in parts:
        return "fake"
    return None

def new_acc():
    return {"formats": defaultdict(int), "sizes": defaultdict(int),
            "modes": defaultdict(int), "jpeg_q": [], "n": 0}

def report(label, a):
    print(f"\n  --- {label} ({a['n']} images) ---")
    print("  Formats:", dict(a["formats"]))
    print("  Modes:", dict(a["modes"]))
    print("  Sizes (WxH: count):")
    for size, count in sorted(a["sizes"].items(), key=lambda x: -x[1]):
        print(f"      {size[0]}x{size[1]}: {count}")
    if a["jpeg_q"]:
        print(f"  JPEG quality est.: min={min(a['jpeg_q'])}, "
              f"max={max(a['jpeg_q'])}, mean={round(statistics.mean(a['jpeg_q']),1)}")

for sf in subfolders:
    folder = parent / sf
    if not folder.exists():
        print(f"[MISSING] {sf}")
        continue

    accs = {"real": new_acc(), "fake": new_acc()}

    for f in folder.rglob("*"):
        if f.suffix.lower() not in exts:
            continue
        split = which_split(f)
        if split is None:
            continue
        try:
            with Image.open(f) as img:
                a = accs[split]
                a["n"] += 1
                a["formats"][img.format] += 1
                a["sizes"][img.size] += 1
                a["modes"][img.mode] += 1
                if img.format == "JPEG":
                    q = jpeg_quality(img)
                    if q is not None:
                        a["jpeg_q"].append(q)
        except Exception as e:
            print(f"  [ERROR] {f}: {e}")

    print(f"\n=== {sf} ===")
    report("real (0_real)", accs["real"])
    report("fake (1_fake)", accs["fake"])


=== biggan ===

  --- real (0_real) (2000 images) ---
  Formats: {'PNG': 2000}
  Modes: {'RGB': 2000}
  Sizes (WxH: count):
      256x256: 2000

  --- fake (1_fake) (2000 images) ---
  Formats: {'PNG': 2000}
  Modes: {'RGB': 2000}
  Sizes (WxH: count):
      256x256: 2000

=== cyclegan ===

  --- real (0_real) (1321 images) ---
  Formats: {'PNG': 1321}
  Modes: {'RGB': 1321}
  Sizes (WxH: count):
      256x256: 1321

  --- fake (1_fake) (1321 images) ---
  Formats: {'PNG': 1321}
  Modes: {'RGB': 1321}
  Sizes (WxH: count):
      256x256: 1321

=== gaugan ===

  --- real (0_real) (5000 images) ---
  Formats: {'PNG': 5000}
  Modes: {'RGB': 4990, 'L': 10}
  Sizes (WxH: count):
      256x256: 5000

  --- fake (1_fake) (5000 images) ---
  Formats: {'PNG': 5000}
  Modes: {'RGB': 5000}
  Sizes (WxH: count):
      256x256: 5000

=== progan ===

  --- real (0_real) (4000 images) ---
  Formats: {'PNG': 4000}
  Modes: {'RGB': 4000}
  Sizes (WxH: count):
      256x256: 4000

  --- fake (1_fake) (

In [ ]:
from PIL import Image, ImageFile
# Optional salvage: let libtiff return partially-decoded truncated files
# instead of raising. Comment out if you'd rather skip bad files entirely.
ImageFile.LOAD_TRUNCATED_IMAGES = True

def compress(original_root, compressed_root, generators):
    valid = {".jpg",".jpeg",".png",".webp",".bmp",".gif",".tiff",".tif"}
    failures = []
    for gen in generators:
        gen_root = os.path.join(original_root, gen)
        out_dir  = os.path.join(compressed_root, gen)
        os.makedirs(out_dir, exist_ok=True)
        for filename in os.listdir(gen_root):
            if os.path.splitext(filename)[1].lower() not in valid:
                continue
            src = os.path.join(gen_root, filename)
            try:
                img = Image.open(src)
                if img.mode != "RGB":
                    img = img.convert("RGB")
                img.load()                       # force decode HERE so errors are catchable
                base = os.path.splitext(filename)[0]
                img.save(os.path.join(out_dir, base + ".jpg"), "JPEG",
                         qtables=FIREFLY_QTABLES,
                         subsampling=FIREFLY_SUBSAMPLING,
                         optimize=True)
            except Exception as e:
                failures.append((src, f"{type(e).__name__}: {e}"))
                print(f"  [SKIP] {filename} -> {e}")
    print(f"\n{gen} done. {len(failures)} failed.")
    return failures

bad = compress("/ceph/tischuet/replication_data/RAISE_1k",
               "/ceph/tischuet/replication_data/New-Generator_RAISE1k_unbiased",
               ["TIFF"])


In [8]:
# --- 1. Capture firefly's exact compression profile ONCE ---
# this step extracts quantization table to match the exact compression of firefly images (all images share the same table)
_ref = Image.open(
    "/home/tischuet/zero-shot-deepfake-detection/datasets/synthbuster/firefly/"
    + next(iter(os.listdir(
        "/home/tischuet/zero-shot-deepfake-detection/datasets/synthbuster/firefly")))
)
FIREFLY_QTABLES = _ref.quantization       # dict {0: [...64...], 1: [...64...]}
FIREFLY_SUBSAMPLING = get_sampling(_ref)  # 2  == 4:2:0


# --- 2. Compress any reference set to that exact profile ---
def compress(original_root, compressed_root, generators):
    valid = {".jpg",".jpeg",".png",".webp",".bmp",".gif",".tiff",".tif"}
    failures = []
    for gen in generators:
        gen_root = os.path.join(original_root, gen)
        out_dir  = os.path.join(compressed_root, gen)
        os.makedirs(out_dir, exist_ok=True)
        for filename in os.listdir(gen_root):
            if os.path.splitext(filename)[1].lower() not in valid:
                continue
            src = os.path.join(gen_root, filename)
            try:
                img = Image.open(src)
                if img.mode != "RGB":
                    img = img.convert("RGB")
                img.load()                       # force decode HERE so errors are catchable
                base = os.path.splitext(filename)[0]
                img.save(os.path.join(out_dir, base + ".jpg"), "JPEG",
                         qtables=FIREFLY_QTABLES,
                         subsampling=FIREFLY_SUBSAMPLING,
                         optimize=True)
            except Exception as e:
                failures.append((src, f"{type(e).__name__}: {e}"))
                print(f"  [SKIP] {filename} -> {e}")
    print(f"\n{gen} done. {len(failures)} failed.")
    return failures

compress("/ceph/tischuet/replication_data/New-Generator",
         "/ceph/tischuet/replication_data/New-Generator_RAISE1k_unbiased",
         ["dalle3", "midjourney-v5"])
compress("/ceph/tischuet/replication_data/RAISE_1k",
         "/ceph/tischuet/replication_data/New-Generator_RAISE1k_unbiased",
         ["TIFF"])


midjourney-v5 done. 0 failed.


TIFFFillStrip: Read error on strip 3962; got 313 bytes, expected 5025.


  [SKIP] r0bf7f938t.TIF -> decoder error -2

TIFF done. 1 failed.


[('/ceph/tischuet/replication_data/RAISE_1k/TIFF/r0bf7f938t.TIF',
  'OSError: decoder error -2')]

In [ ]:
# ############################################################################
# SUPERSEDED — split 2 WITH build-time pairwise resize.
# Kept for reference; resizing now happens at TEST TIME instead.
# Active version = the compression-only cell below.
# ############################################################################
#
# # ============================================================================
# # SPLIT 2 — real COCO2017 (anchor, untouched) + flux/sdxl/sd3
# #   reals : dominant-qtable subset copied byte-identical
# #   fakes : resized to their COCO partner's exact size (cover+crop, bicubic),
# #           then encoded ONCE with the reals' qtable + subsampling
# # ============================================================================
# import shutil, math
# from pathlib import Path
# from PIL import Image
# from PIL.JpegImagePlugin import get_sampling
#
# SRC_ROOT   = Path("/ceph/tischuet/replication_data/New-Generator")
# OUT_ROOT   = Path("/ceph/tischuet/replication_data/New-Generator_COCO17_unbiased")
# GENERATORS = ["flux", "sdxl", "sd3"]
# IMG_EXTS   = {".jpg", ".jpeg", ".png", ".webp", ".bmp", ".gif", ".tiff", ".tif"}
# BICUBIC    = Image.Resampling.BICUBIC
#
# def qkey(im):
#     return tuple(tuple(t) for t in im.quantization.values())
#
# def resize_cover_crop(img, tw, th):
#     """Scale to fully cover (tw,th) with one bicubic resample, then center-crop.
#     No aspect distortion, no padding."""
#     w, h = img.size
#     s = max(tw / w, th / h)
#     nw, nh = max(tw, math.ceil(w * s)), max(th, math.ceil(h * s))
#     img = img.resize((nw, nh), BICUBIC)
#     l, t = (nw - tw) // 2, (nh - th) // 2
#     return img.crop((l, t, l + tw, t + th))
#
# # ---- 1. reference profile + pair sizes: dominant qtable group of the reals --
# groups, sizes = {}, {}
# for f in sorted((SRC_ROOT / "real").iterdir()):
#     if f.suffix.lower() not in IMG_EXTS:
#         continue
#     with Image.open(f) as im:                      # header-only, no decode
#         if im.format != "JPEG":
#             continue
#         groups.setdefault(qkey(im), []).append(f)
#         sizes[f] = im.size
#
# REF_KEY   = max(groups, key=lambda k: len(groups[k]))
# REF_FILES = sorted(groups[REF_KEY])
# with Image.open(REF_FILES[0]) as im:
#     REF_QTABLES = [list(im.quantization[k]) for k in sorted(im.quantization)]
#     REF_SUB     = get_sampling(im)
# pair_size = {f.stem: sizes[f] for f in REF_FILES}   # COCO id -> (w, h)
#
# print(f"reference group: {len(REF_FILES)} reals | sub={REF_SUB} | "
#       f"luma row0={REF_QTABLES[0][:8]}")
#
# # ---- 2. reals: byte-identical copy ------------------------------------------
# out_real = OUT_ROOT / "real"
# out_real.mkdir(parents=True, exist_ok=True)
# for i, f in enumerate(REF_FILES, 1):
#     shutil.copy2(f, out_real / f.name)
#     if i % 1000 == 0:
#         print(f"  real {i}/{len(REF_FILES)}")
# print(f"[real] {len(REF_FILES)} copied")
#
# # ---- 3. fakes: pairwise resize -> single encode with reference profile ------
# failures = []
# for gen in GENERATORS:
#     out_gen = OUT_ROOT / gen
#     out_gen.mkdir(parents=True, exist_ok=True)
#     by_stem = {f.stem: f for f in (SRC_ROOT / gen).iterdir()
#                if f.suffix.lower() in IMG_EXTS}
#     written = missing = 0
#     for stem, (tw, th) in pair_size.items():
#         src = by_stem.get(stem)
#         if src is None:
#             missing += 1
#             continue
#         try:
#             with Image.open(src) as img:
#                 img.load()
#                 if img.mode != "RGB":
#                     img = img.convert("RGB")
#                 out = resize_cover_crop(img, tw, th)
#                 out.save(out_gen / f"{stem}.jpg", "JPEG",
#                          qtables=REF_QTABLES, subsampling=REF_SUB, optimize=True)
#             written += 1
#             if written % 500 == 0:
#                 print(f"  [{gen}] {written}")
#         except Exception as e:
#             failures.append((str(src), f"{type(e).__name__}: {e}"))
#             print(f"  [SKIP] {gen}/{src.name} -> {e}")
#     print(f"[{gen}] wrote {written}, missing {missing}")
# print(f"failures: {len(failures)}")
#
# # ---- 4. verify: same table, same subsampling, size == paired real -----------
# for gen in GENERATORS:
#     checked = 0
#     for f in (OUT_ROOT / gen).glob("*.jpg"):
#         with Image.open(f) as im:
#             assert qkey(im) == REF_KEY, f"table mismatch: {f}"
#             assert get_sampling(im) == REF_SUB, f"subsampling mismatch: {f}"
#             assert im.size == tuple(pair_size[f.stem]), f"size mismatch: {f}"
#         checked += 1
#     print(f"[{gen}] verified {checked} ✓")
# print("split 2 done.")


In [1]:
# ============================================================================
# SPLIT 2 (ACTIVE) — compression-only, NO resize — real COCO2017 + flux/sdxl/sd3
#   real = anchor: the 3530-image dominant-qtable subset, copied verbatim
#   flux/sdxl/sd3: encoded ONCE at NATIVE size with the reals' qtable + subsampling
#   Sizes are left untouched -> resizing happens at TEST TIME.
# ============================================================================
import shutil
from pathlib import Path
from collections import Counter
from PIL import Image
from PIL.JpegImagePlugin import get_sampling

SRC_ROOT   = Path("/ceph/tischuet/replication_data/New-Generator")
OUT_ROOT   = Path("/ceph/tischuet/replication_data/New-Generator_COCO17_unbiased")
GENERATORS = ["flux", "sdxl", "sd3"]
IMG_EXTS   = {".jpg", ".jpeg", ".png", ".webp", ".bmp", ".gif", ".tiff", ".tif"}

def qkey(im):
    """Hashable identity of a JPEG's quantization tables."""
    return tuple(tuple(t) for t in im.quantization.values())

# ---- 1. reference profile: dominant qtable group of the reals (the 3530) ---
groups = {}
for f in sorted((SRC_ROOT / "real").iterdir()):
    if f.suffix.lower() not in IMG_EXTS:
        continue
    with Image.open(f) as im:                      # header-only, no decode
        if im.format != "JPEG":
            continue
        groups.setdefault(qkey(im), []).append(f)

REF_KEY   = max(groups, key=lambda k: len(groups[k]))
REF_FILES = sorted(groups[REF_KEY])
with Image.open(REF_FILES[0]) as im:
    REF_QTABLES = [list(im.quantization[k]) for k in sorted(im.quantization)]
    REF_SUB     = get_sampling(im)
print(f"reference group: {len(REF_FILES)} reals | {len(groups)} distinct tables | "
      f"sub={REF_SUB} | luma row0={REF_QTABLES[0][:8]}")

# ---- 2. reals: byte-identical copy (they ARE the reference) ----------------
out_real = OUT_ROOT / "real"
out_real.mkdir(parents=True, exist_ok=True)
for i, f in enumerate(REF_FILES, 1):
    shutil.copy2(f, out_real / f.name)
    if i % 1000 == 0:
        print(f"  real {i}/{len(REF_FILES)}")
stems = {f.stem for f in REF_FILES}
print(f"[real] {len(REF_FILES)} copied")

# ---- 3. fakes: single encode at NATIVE size with the reference profile -----
failures = []
for gen in GENERATORS:
    out_gen = OUT_ROOT / gen
    out_gen.mkdir(parents=True, exist_ok=True)
    by_stem = {f.stem: f for f in (SRC_ROOT / gen).iterdir()
               if f.suffix.lower() in IMG_EXTS}
    written = missing = 0
    for s in sorted(stems):
        src = by_stem.get(s)
        if src is None:
            missing += 1
            continue
        try:
            with Image.open(src) as img:
                img.load()
                if img.mode != "RGB":
                    img = img.convert("RGB")
                img.save(out_gen / f"{s}.jpg", "JPEG",       # NO resize
                         qtables=REF_QTABLES,                # never pass `quality` too
                         subsampling=REF_SUB, optimize=True)
            written += 1
            if written % 500 == 0:
                print(f"  [{gen}] {written}")
        except Exception as e:
            failures.append((str(src), f"{type(e).__name__}: {e}"))
            print(f"  [SKIP] {gen}/{src.name} -> {e}")
    print(f"[{gen}] wrote {written}, missing {missing}")
print(f"failures: {len(failures)}")

# ---- 4. verify: one profile everywhere; sizes stay NATIVE (differ by design)
for folder in ["real", *GENERATORS]:
    sizes, n = Counter(), 0
    for f in (OUT_ROOT / folder).glob("*"):
        with Image.open(f) as im:
            assert qkey(im) == REF_KEY, f"table mismatch: {f}"
            assert get_sampling(im) == REF_SUB, f"subsampling mismatch: {f}"
            sizes[im.size] += 1
        n += 1
    print(f"[{folder}] {n} ✓ profile | native sizes: {sizes.most_common(3)}")
print("split 2 (compression-only) done.")


reference group: 3530 reals | 4 distinct tables | sub=0 | luma row0=[1, 1, 1, 1, 2, 3, 4, 5]
  real 1000/3530
  real 2000/3530
  real 3000/3530
[real] 3530 copied
  [flux] 500
  [flux] 1000
  [flux] 1500
  [flux] 2000
  [flux] 2500
  [flux] 3000
  [flux] 3500
[flux] wrote 3530, missing 0
  [sdxl] 500
  [sdxl] 1000
  [sdxl] 1500
  [sdxl] 2000
  [sdxl] 2500
  [sdxl] 3000
  [sdxl] 3500
[sdxl] wrote 3530, missing 0
  [sd3] 500
  [sd3] 1000
  [sd3] 1500
  [sd3] 2000
  [sd3] 2500
  [sd3] 3000
  [sd3] 3500
[sd3] wrote 3530, missing 0
failures: 0
[real] 3530 ✓ profile | native sizes: [((640, 480), 569), ((640, 427), 443), ((500, 375), 220)]
[flux] 3530 ✓ profile | native sizes: [((512, 512), 3530)]
[sdxl] 3530 ✓ profile | native sizes: [((1024, 1024), 3530)]
[sd3] 3530 ✓ profile | native sizes: [((512, 512), 3530)]
split 2 (compression-only) done.


In [ ]:
# ############################################################################
# SUPERSEDED — split 1 WITH build-time pairwise resize (3 real variants).
# Kept for reference; resizing now happens at TEST TIME instead.
# Last run: 999/1000 reals, 1 corrupt TIFF skipped, 5/2997 outputs upsampled
#           (all real_firefly, max scale 1.152 @ r165355c0t).
# Active version = the compression-only cell below.
# ############################################################################
#
# # ============================================================================
# # SPLIT 1 — RAISE_1k vs firefly/dalle3/midjourney-v5   (MIRROR of split 2)
# #   split 2: real = anchor (verbatim); FAKES cover-crop-resized to real's exact
# #            per-stem size, then real's qtable.
# #   split 1: firefly = anchor (verbatim); REALS cover-crop-resized (SAME op) to
# #            each generator's EXACT per-stem size (paired by RAISE id), then
# #            firefly's qtable.  dalle3/MJ: encode-once at native size.
# #   3 real variants because each generator has a different size for the same id.
# # NOTE: decodes ~1000 large TIFFs once and writes 3 JPEGs each — slow on ceph.
# # ============================================================================
# import shutil, math
# from pathlib import Path
# from collections import defaultdict
# from PIL import Image, ImageFile
# from PIL.JpegImagePlugin import get_sampling
# ImageFile.LOAD_TRUNCATED_IMAGES = True          # salvage the one flaky RAISE TIFF
#
# SRC_NG     = Path("/ceph/tischuet/replication_data/New-Generator")
# SRC_RAISE  = Path("/ceph/tischuet/replication_data/RAISE_1k/TIFF")
# OUT_ROOT   = Path("/ceph/tischuet/replication_data/New-Generator_RAISE1k_unbiased")
# GENERATORS = ["firefly", "dalle3", "midjourney-v5"]
# IMG_EXTS   = {".jpg", ".jpeg", ".png", ".webp", ".bmp", ".gif", ".tiff", ".tif"}
# BICUBIC    = Image.Resampling.BICUBIC
#
# def qkey(im):
#     return tuple(tuple(t) for t in im.quantization.values())
#
# def resize_cover_crop(img, tw, th):             # <-- IDENTICAL to split 2 (cell 11)
#     """Scale to fully cover (tw,th) with one bicubic resample, then center-crop.
#     No aspect distortion, no padding."""
#     w, h = img.size
#     s = max(tw / w, th / h)
#     nw, nh = max(tw, math.ceil(w * s)), max(th, math.ceil(h * s))
#     img = img.resize((nw, nh), BICUBIC)
#     l, t = (nw - tw) // 2, (nh - th) // 2
#     return img.crop((l, t, l + tw, t + th))
#
# # ---- 1. reference profile from firefly (the JPEG anchor) -------------------
# ff_files = sorted(f for f in (SRC_NG / "firefly").iterdir() if f.suffix.lower() in IMG_EXTS)
# with Image.open(ff_files[0]) as im:
#     assert im.format == "JPEG", "firefly is not JPEG?!"
#     REF_QTABLES = [list(im.quantization[k]) for k in sorted(im.quantization)]
#     REF_SUB     = get_sampling(im)
#     REF_KEY     = qkey(im)
# print(f"firefly profile | sub={REF_SUB} | luma row0={REF_QTABLES[0][:8]}")
#
# # ---- 2. fakes: firefly verbatim; dalle3/MJ encode-once at native size ------
# #        record each generator's EXACT per-stem size for pairing the reals.
# gen_size, failures = {}, []
# for gen in GENERATORS:
#     out_gen = OUT_ROOT / gen
#     out_gen.mkdir(parents=True, exist_ok=True)
#     gen_size[gen] = {}
#     n = 0
#     for f in sorted((SRC_NG / gen).iterdir()):
#         if f.suffix.lower() not in IMG_EXTS:
#             continue
#         try:
#             if gen == "firefly":
#                 with Image.open(f) as im:
#                     gen_size[gen][f.stem] = im.size          # header only
#                 shutil.copy2(f, out_gen / f.name)            # verbatim anchor
#             else:
#                 with Image.open(f) as img:
#                     img.load()
#                     if img.mode != "RGB":
#                         img = img.convert("RGB")
#                     gen_size[gen][f.stem] = img.size
#                     img.save(out_gen / f"{f.stem}.jpg", "JPEG",
#                              qtables=REF_QTABLES, subsampling=REF_SUB, optimize=True)
#             n += 1
#         except Exception as e:
#             failures.append((str(f), f"{type(e).__name__}: {e}"))
#             print(f"  [SKIP] {gen}/{f.name} -> {e}")
#     print(f"[{gen}] {n} written")
#
# # ---- 3. reals: downsample RAISE -> each generator's EXACT per-stem size ----
# #        (same resize_cover_crop op as split 2, then firefly-profile JPEG)
# VARIANTS = {"real_firefly": "firefly", "real_dalle3": "dalle3", "real_mjv5": "midjourney-v5"}
# for v in VARIANTS:
#     (OUT_ROOT / v).mkdir(parents=True, exist_ok=True)
#
# upsampled = defaultdict(list)                    # variant -> [(stem, scale), ...] where scale > 1
# tiffs = sorted(f for f in SRC_RAISE.iterdir() if f.suffix.lower() in {".tif", ".tiff"})
# done = 0
# for f in tiffs:
#     try:
#         with Image.open(f) as img:
#             img.load()                                       # decode the big TIFF ONCE
#             if img.mode != "RGB":
#                 img = img.convert("RGB")
#             w, h = img.size
#             for v, gen in VARIANTS.items():
#                 tw, th = gen_size[gen][f.stem]               # EXACT paired counterpart size
#                 s = max(tw / w, th / h)                      # cover scale; >1 == upsample
#                 if s > 1.0:
#                     upsampled[v].append((f.stem, round(s, 4)))
#                 resize_cover_crop(img, tw, th).save(         # SAME op as split 2
#                     OUT_ROOT / v / f"{f.stem}.jpg", "JPEG",
#                     qtables=REF_QTABLES, subsampling=REF_SUB, optimize=True)
#         done += 1
#         if done % 100 == 0:
#             print(f"  [reals] {done}/{len(tiffs)}")
#     except Exception as e:
#         failures.append((str(f), f"{type(e).__name__}: {e}"))
#         print(f"  [SKIP] {f.name} -> {e}")
# print(f"[reals] {done}/{len(tiffs)} done | total failures: {len(failures)}")
#
# # ---- upsampling report -----------------------------------------------------
# tot_up = sum(len(u) for u in upsampled.values())
# print(f"\nUPSAMPLED real images (cover scale > 1) — {tot_up} of {done * len(VARIANTS)} outputs:")
# for v in VARIANTS:
#     u = sorted(upsampled[v], key=lambda x: -x[1])
#     mx = f" | max scale {u[0][1]} ({u[0][0]})" if u else ""
#     print(f"  {v}: {len(u)}/{done}{mx}")
#
# # ---- 4. verify: one profile everywhere; real_X size == generator-X size ----
# for gen in GENERATORS:
#     for f in (OUT_ROOT / gen).glob("*"):                     # firefly kept .png name
#         with Image.open(f) as im:
#             assert qkey(im) == REF_KEY and get_sampling(im) == REF_SUB, f
#     print(f"[{gen}] profile ok ✓")
# for v, gen in VARIANTS.items():
#     for f in (OUT_ROOT / v).glob("*.jpg"):
#         with Image.open(f) as im:
#             assert qkey(im) == REF_KEY and get_sampling(im) == REF_SUB, f
#             assert im.size == tuple(gen_size[gen][f.stem]), f"size != {gen}: {f}"
#     print(f"[{v}] exact size-match to {gen} ✓")
# print("split 1 done.")


In [1]:
# ============================================================================
# SPLIT 1 (ACTIVE) — compression-only, NO resize — RAISE_1k + firefly/dalle3/mjv5
#   firefly = anchor: copied verbatim (defines qtable + subsampling)
#   RAISE real + dalle3 + MJ: encoded ONCE at NATIVE size with firefly's profile
#   ONE shared real/ folder for all three generators; strict stem pairing.
#   Sizes are left untouched -> resizing happens at TEST TIME.
# NOTE: decodes ~1000 large TIFFs once — slow on ceph.
# ============================================================================
import shutil
from pathlib import Path
from collections import Counter
from PIL import Image, ImageFile
from PIL.JpegImagePlugin import get_sampling
ImageFile.LOAD_TRUNCATED_IMAGES = True          # salvage the one flaky RAISE TIFF

SRC_NG     = Path("/ceph/tischuet/replication_data/New-Generator")
SRC_RAISE  = Path("/ceph/tischuet/replication_data/RAISE_1k/TIFF")
OUT_ROOT   = Path("/ceph/tischuet/replication_data/New-Generator_RAISE1k_unbiased")
GENERATORS = ["firefly", "dalle3", "midjourney-v5"]
IMG_EXTS   = {".jpg", ".jpeg", ".png", ".webp", ".bmp", ".gif", ".tiff", ".tif"}

def qkey(im):
    """Hashable identity of a JPEG's quantization tables."""
    return tuple(tuple(t) for t in im.quantization.values())

# ---- 1. reference profile from firefly (the JPEG anchor) -------------------
ff_files = sorted(f for f in (SRC_NG / "firefly").iterdir() if f.suffix.lower() in IMG_EXTS)
with Image.open(ff_files[0]) as im:
    assert im.format == "JPEG", "firefly is not JPEG?!"
    REF_QTABLES = [list(im.quantization[k]) for k in sorted(im.quantization)]
    REF_SUB     = get_sampling(im)
    REF_KEY     = qkey(im)
print(f"firefly profile | sub={REF_SUB} | luma row0={REF_QTABLES[0][:8]}")

# ---- 2. reals: RAISE TIFF -> firefly-profile JPEG at NATIVE size -----------
#        encoded FIRST so ok_stems can gate the fakes -> strict pairing.
out_real = OUT_ROOT / "real"
out_real.mkdir(parents=True, exist_ok=True)
tiffs = sorted(f for f in SRC_RAISE.iterdir() if f.suffix.lower() in {".tif", ".tiff"})
ok_stems, failures = set(), []
for i, f in enumerate(tiffs, 1):
    try:
        with Image.open(f) as img:
            img.load()
            if img.mode != "RGB":
                img = img.convert("RGB")
            img.save(out_real / f"{f.stem}.jpg", "JPEG",     # NO resize
                     qtables=REF_QTABLES, subsampling=REF_SUB, optimize=True)
        ok_stems.add(f.stem)
    except Exception as e:
        failures.append((str(f), f"{type(e).__name__}: {e}"))
        print(f"  [SKIP] {f.name} -> {e}")
    if i % 100 == 0:
        print(f"  [real] {i}/{len(tiffs)}")
print(f"[real] {len(ok_stems)}/{len(tiffs)} written (shared by all 3 generators)")

# ---- 3. fakes, restricted to stems that have a real -> strict pairing ------
for gen in GENERATORS:
    out_gen = OUT_ROOT / gen
    out_gen.mkdir(parents=True, exist_ok=True)
    n = skipped = 0
    for f in sorted((SRC_NG / gen).iterdir()):
        if f.suffix.lower() not in IMG_EXTS:
            continue
        if f.stem not in ok_stems:               # no real counterpart -> drop
            skipped += 1
            continue
        try:
            if gen == "firefly":
                shutil.copy2(f, out_gen / f.name)            # verbatim anchor
            else:
                with Image.open(f) as img:
                    img.load()
                    if img.mode != "RGB":
                        img = img.convert("RGB")
                    img.save(out_gen / f"{f.stem}.jpg", "JPEG",   # NO resize
                             qtables=REF_QTABLES, subsampling=REF_SUB, optimize=True)
            n += 1
        except Exception as e:
            failures.append((str(f), f"{type(e).__name__}: {e}"))
            print(f"  [SKIP] {gen}/{f.name} -> {e}")
    print(f"[{gen}] {n} written, {skipped} dropped (no real counterpart)")
print(f"failures: {len(failures)}")

# ---- 4. verify: one profile everywhere; sizes stay NATIVE (differ by design)
for folder in ["real", *GENERATORS]:
    sizes, n = Counter(), 0
    for f in (OUT_ROOT / folder).glob("*"):      # firefly kept its .png name
        with Image.open(f) as im:
            assert qkey(im) == REF_KEY, f"table mismatch: {f}"
            assert get_sampling(im) == REF_SUB, f"subsampling mismatch: {f}"
            sizes[im.size] += 1
        n += 1
    print(f"[{folder}] {n} ✓ profile | native sizes: {sizes.most_common(3)}")
print("split 1 (compression-only) done.")


firefly profile | sub=2 | luma row0=[2, 1, 1, 2, 2, 4, 5, 6]
  [real] 100/1000
  [real] 200/1000
  [real] 300/1000


TIFFFillStrip: Read error on strip 3962; got 313 bytes, expected 5025.


  [SKIP] r0bf7f938t.TIF -> decoder error -2
  [real] 400/1000
  [real] 500/1000
  [real] 600/1000
  [real] 700/1000
  [real] 800/1000
  [real] 900/1000
  [real] 1000/1000
[real] 999/1000 written (shared by all 3 generators)
[firefly] 999 written, 1 dropped (no real counterpart)
[dalle3] 999 written, 1 dropped (no real counterpart)
[midjourney-v5] 999 written, 1 dropped (no real counterpart)
failures: 1
[real] 999 ✓ profile | native sizes: [((4928, 3264), 571), ((4288, 2848), 177), ((3264, 4928), 157)]
[firefly] 999 ✓ profile | native sizes: [((2304, 1792), 271), ((1792, 2304), 252), ((2048, 2048), 243)]
[dalle3] 999 ✓ profile | native sizes: [((1024, 1024), 840), ((1792, 1024), 85), ((1024, 1792), 74)]
[midjourney-v5] 999 ✓ profile | native sizes: [((1360, 896), 571), ((1344, 896), 182), ((896, 1360), 157)]
split 1 (compression-only) done.
